
# 04. Phân tích khám phá dữ liệu tuyển dụng IT (EDA)

## 1. Mục tiêu

Phân tích dữ liệu tuyển dụng IT tại Việt Nam nhằm khám phá đặc điểm thị trường tuyển dụng, nhu cầu kỹ năng, xu hướng đăng tuyển, mức lương được công khai và dự báo ngắn hạn số lượng tin tuyển dụng.

## 2. Dữ liệu sử dụng

Dữ liệu được lấy từ Data Warehouse đã xây dựng qua quy trình ETL, bao gồm:

- FACT_JOB_POSTINGS: thông tin tin tuyển dụng.
- FACT_JOB_SKILLS: kỹ năng được yêu cầu trong tin tuyển dụng.
- DIM_CITY: thành phố.
- DIM_ROLE: vai trò IT.
- DIM_LEVEL: cấp bậc.
- DIM_SKILL: kỹ năng.
- DIM_DATE: thời gian.
- DIM_COMPANY: công ty tuyển dụng.

Tổng dữ liệu đầu vào gồm 7.232 tin tuyển dụng.

## 3. Nội dung phân tích

- Tổng quan thị trường tuyển dụng IT.
- Phân tích nhu cầu kỹ năng.
- Phân tích xu hướng tuyển dụng theo thời gian.
- Dự báo số lượng tin tuyển dụng ngắn hạn.
- Phân tích mức lương tuyển dụng.


In [ ]:

# =========================================================
# 04. PHAN TICH KHAM PHA DU LIEU TUYEN DUNG IT (EDA)
# =========================================================

# 1. KHAI BAO THU VIEN

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path


# =========================================================
# 2. CAU HINH
# =========================================================

pd.set_option("display.max_columns", None)

plt.rcParams["font.sans-serif"] = ["Arial"]
plt.rcParams["axes.unicode_minus"] = False

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


# =========================================================
# 3. XAC DINH DUONG DAN DU LIEU
# =========================================================

DATA_DIR = Path("data/warehouse")

if not DATA_DIR.exists():
    DATA_DIR = Path("../data/warehouse")

PROJECT_ROOT = DATA_DIR.parent.parent

CHART_DIR = PROJECT_ROOT / "results" / "charts"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"

CHART_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# 4. DOC DU LIEU DATA WAREHOUSE
# =========================================================

jobs = pd.read_csv(
    DATA_DIR / "fact_job_postings.csv",
    dtype={"job_id": str}
)

fact_skills = pd.read_csv(
    DATA_DIR / "fact_job_skills.csv",
    dtype={"job_id": str}
)

dim_role = pd.read_csv(DATA_DIR / "dim_role.csv")
dim_city = pd.read_csv(DATA_DIR / "dim_city.csv")
dim_skill = pd.read_csv(DATA_DIR / "dim_skill.csv")
dim_level = pd.read_csv(DATA_DIR / "dim_level.csv")
dim_date = pd.read_csv(DATA_DIR / "dim_date.csv")
dim_company = pd.read_csv(DATA_DIR / "dim_company.csv")


# Chuan hoa job_id
jobs["job_id"] = jobs["job_id"].astype(str).str.strip()
fact_skills["job_id"] = (
    fact_skills["job_id"].astype(str).str.strip()
)


# =========================================================
# 5. CAU HINH XUAT BIEU DO
# =========================================================

chart_names = [
    "jobs_by_city",
    "jobs_by_role",
    "top_15_skills",
    "skills_by_role_heatmap",
    "skill_requirement_rate",
    "recruitment_trend",
    "salary_by_role",
    "salary_by_level",
    "salary_disclosure_pie"
]

chart_index = 0

_original_show = plt.show


def save_and_show(*args, **kwargs):
    global chart_index

    if chart_index < len(chart_names):

        filename = chart_names[chart_index]

        plt.gcf().savefig(
            CHART_DIR / f"{filename}.png",
            dpi=300,
            bbox_inches="tight"
        )

        print(f"Da xuat: {filename}.png")

        chart_index += 1

    _original_show(*args, **kwargs)


plt.show = save_and_show


# =========================================================
# 6. KIEM TRA DU LIEU
# =========================================================

print("=" * 50)
print("TONG QUAN DATA WAREHOUSE")
print("=" * 50)

print("So tin tuyen dung:", len(jobs))
print("So job_id duy nhat:", jobs["job_id"].nunique())
print("So dong ky nang:", len(fact_skills))
print("So vi tri IT:", len(dim_role))
print("So thanh pho:", len(dim_city))
print("So ky nang:", len(dim_skill))
print("So cap bac:", len(dim_level))
print("So cong ty:", len(dim_company))

print("\nCac cot Fact Job Postings:")
print(jobs.columns.tolist())

print("\nCac cot Fact Job Skills:")
print(fact_skills.columns.tolist())

print("\nKiem tra job_id trung lap:")
print("FACT_JOB_POSTINGS:", jobs["job_id"].duplicated().sum())

display(jobs.head())

In [ ]:
# =========================================================
# PHAN TICH 1: TONG QUAN THI TRUONG
# =========================================================

# Ghep du lieu thanh pho, vai tro va cong ty

tong_quan = jobs.merge(
    dim_city,
    on="city_id",
    how="left"
)

tong_quan = tong_quan.merge(
    dim_role,
    on="role_id",
    how="left"
)

tong_quan = tong_quan.merge(
    dim_company,
    on="company_id",
    how="left"
)


# ---------------------------------------------------------
# 1.1. CAC CHI SO TONG QUAN
# ---------------------------------------------------------

tong_so_tin = tong_quan["job_id"].nunique()

# Khong tinh cong ty Unknown co company_id = 0
tong_so_cong_ty = (
    tong_quan.loc[
        tong_quan["company_id"] != 0,
        "company_id"
    ].nunique()
)

luong_trung_vi = tong_quan["salary_avg"].median()

print("\n===== CAC CHI SO TONG QUAN =====")

print(f"Tong so tin tuyen dung: {tong_so_tin:,}")
print(f"Tong so cong ty xac dinh duoc: {tong_so_cong_ty:,}")

if pd.notna(luong_trung_vi):
    print(
        f"Luong trung vi: {luong_trung_vi:.2f} "
        "trieu VND/thang"
    )
else:
    print("Khong co du lieu luong hop le.")


# ---------------------------------------------------------
# 1.2. TIN TUYEN DUNG THEO THANH PHO
# ---------------------------------------------------------

tin_theo_thanh_pho = (
    tong_quan.groupby("city_name")["job_id"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="so_tin")
)

print("\n===== TIN TUYEN DUNG THEO THANH PHO =====")

display(tin_theo_thanh_pho)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=tin_theo_thanh_pho,
    x="so_tin",
    y="city_name",
    color="steelblue"
)

plt.title("So tin tuyen dung theo thanh pho")
plt.xlabel("So tin tuyen dung")
plt.ylabel("Thanh pho")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 1.3. TIN TUYEN DUNG THEO VAI TRO
# ---------------------------------------------------------

tin_theo_vai_tro = (
    tong_quan.groupby("role_name")["job_id"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="so_tin")
)

print("\n===== TIN TUYEN DUNG THEO VAI TRO =====")

display(tin_theo_vai_tro)

plt.figure(figsize=(12, 7))

sns.barplot(
    data=tin_theo_vai_tro,
    x="so_tin",
    y="role_name",
    color="steelblue"
)

plt.title("So tin tuyen dung theo vai tro")
plt.xlabel("So tin tuyen dung")
plt.ylabel("Vai tro IT")

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# PHAN TICH 2: NHU CAU KY NANG
# =========================================================

# Ghep du lieu ky nang va vai tro

ky_nang = fact_skills.merge(
    dim_skill,
    on="skill_id",
    how="left"
)

ky_nang = ky_nang.merge(
    dim_role,
    on="role_id",
    how="left"
)


# ---------------------------------------------------------
# 2.1. TOP 15 KY NANG DUOC YEU CAU NHIEU NHAT
# ---------------------------------------------------------

top_15_ky_nang = (
    ky_nang.groupby("skill_name")["job_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(15)
    .reset_index(name="so_tin")
)

print("\n===== TOP 15 KY NANG DUOC YEU CAU NHIEU NHAT =====")

display(top_15_ky_nang)

plt.figure(figsize=(12, 7))

sns.barplot(
    data=top_15_ky_nang,
    x="so_tin",
    y="skill_name",
    color="steelblue"
)

plt.title("Top 15 ky nang duoc yeu cau nhieu nhat")
plt.xlabel("So tin tuyen dung")
plt.ylabel("Ky nang")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 2.2. KY NANG THEO VAI TRO
# ---------------------------------------------------------

top_10_ky_nang = (
    top_15_ky_nang.head(10)["skill_name"].tolist()
)

ky_nang_theo_vai_tro = (
    ky_nang[
        ky_nang["skill_name"].isin(top_10_ky_nang)
    ]
    .groupby(["role_name", "skill_name"])["job_id"]
    .nunique()
    .reset_index(name="so_tin")
)

bang_ky_nang_theo_vai_tro = (
    ky_nang_theo_vai_tro.pivot(
        index="role_name",
        columns="skill_name",
        values="so_tin"
    ).fillna(0)
)

print("\n===== KY NANG THEO VAI TRO =====")

display(bang_ky_nang_theo_vai_tro)

plt.figure(figsize=(16, 8))

sns.heatmap(
    bang_ky_nang_theo_vai_tro,
    annot=True,
    fmt=".0f",
    cmap="Blues"
)

plt.title("So tin tuyen dung yeu cau ky nang theo vai tro")
plt.xlabel("Ky nang")
plt.ylabel("Vai tro")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 2.3. TY LE TIN TUYEN DUNG YEU CAU KY NANG
# ---------------------------------------------------------

tong_so_tin_tuyen_dung = jobs["job_id"].nunique()

ty_le_ky_nang = (
    ky_nang.groupby("skill_name")["job_id"]
    .nunique()
    .reset_index(name="so_tin")
)

ty_le_ky_nang["ty_le"] = (
    ty_le_ky_nang["so_tin"]
    / tong_so_tin_tuyen_dung
    * 100
)

ty_le_ky_nang = (
    ty_le_ky_nang
    .sort_values("ty_le", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

print("\n===== TY LE TIN TUYEN DUNG YEU CAU KY NANG =====")

display(ty_le_ky_nang)

plt.figure(figsize=(12, 7))

sns.barplot(
    data=ty_le_ky_nang,
    x="ty_le",
    y="skill_name",
    color="steelblue"
)

plt.title("Ty le tin tuyen dung yeu cau ky nang")
plt.xlabel("Ty le (%)")
plt.ylabel("Ky nang")

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# PHAN TICH 3: XU HUONG TUYEN DUNG THEO THOI GIAN
# =========================================================

# Ghep du lieu ngay voi tin tuyen dung

xu_huong = jobs.merge(
    dim_date,
    on="date_id",
    how="left"
)

# Chuyen ngay dang ve datetime
xu_huong["full_date"] = pd.to_datetime(
    xu_huong["full_date"],
    errors="coerce"
)

# Loai cac ban ghi khong co ngay khoi phan tich thoi gian
xu_huong = xu_huong.dropna(
    subset=["full_date"]
)


# ---------------------------------------------------------
# 3.1. XU HUONG SO TIN TUYEN DUNG THEO NGAY
# ---------------------------------------------------------

tin_theo_thoi_gian = (
    xu_huong.groupby("full_date")["job_id"]
    .nunique()
    .reset_index(name="so_tin")
    .sort_values("full_date")
)

print("\n===== XU HUONG SO TIN TUYEN DUNG THEO THOI GIAN =====")

display(tin_theo_thoi_gian)

plt.figure(figsize=(15, 6))

sns.lineplot(
    data=tin_theo_thoi_gian,
    x="full_date",
    y="so_tin",
    color="steelblue"
)

plt.title("Xu huong so tin tuyen dung theo thoi gian")
plt.xlabel("Ngay dang tuyen")
plt.ylabel("So tin tuyen dung")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# =========================================================
# PHAN TICH 3.2: DU BAO SO LUONG TIN TUYEN DUNG
# =========================================================

# Phuong phap: Moving Average (MA) voi cua so 3 tuan
# Du bao 2 tuan tiep theo cua chuoi du lieu.
# Danh gia bang MAE va MAPE tren 2 tuan cuoi.


# ---------------------------------------------------------
# 3.2.1. DOC DU LIEU THEO TUAN
# ---------------------------------------------------------

weekly_file = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "it_jobs_weekly.csv"
)

weekly = pd.read_csv(weekly_file)

weekly["post_date"] = pd.to_datetime(
    weekly["post_date"],
    errors="coerce"
)

weekly["job_count"] = pd.to_numeric(
    weekly["job_count"],
    errors="coerce"
)

weekly = (
    weekly.dropna(subset=["post_date", "job_count"])
    .sort_values("post_date")
    .reset_index(drop=True)
)

if len(weekly) < 5:
    raise ValueError(
        "Khong du du lieu theo tuan de danh gia MA 3 tuan."
    )

print("\n===== DU LIEU TUYEN DUNG THEO TUAN =====")

display(weekly)


# ---------------------------------------------------------
# 3.2.2. DANH GIA MO HINH
# ---------------------------------------------------------

# Su dung 3 tuan truoc de du bao tung tuan trong 2 tuan cuoi.
# Tuan thu hai duoc du bao theo cach rolling one-step,
# su dung gia tri thuc te cua tuan thu nhat trong giai doan test.

test_size = 2
train_end = len(weekly) - test_size

actual = weekly["job_count"].iloc[-test_size:].to_numpy()

predicted = []

history = weekly["job_count"].iloc[:train_end].tolist()

for i in range(test_size):

    prediction = np.mean(history[-3:])

    predicted.append(prediction)

    # Them gia tri thuc te de du bao tuan tiep theo
    history.append(actual[i])

predicted = np.array(predicted)

mae = np.mean(
    np.abs(actual - predicted)
)

# Tranh chia cho 0 khi tinh MAPE
non_zero_mask = actual != 0

if non_zero_mask.any():

    mape = np.mean(
        np.abs(
            (actual[non_zero_mask] - predicted[non_zero_mask])
            / actual[non_zero_mask]
        )
    ) * 100

else:
    mape = np.nan


bang_danh_gia = pd.DataFrame({
    "tuan": weekly["post_date"].iloc[-test_size:].to_numpy(),
    "thuc_te": actual,
    "du_bao": np.round(predicted, 2)
})

print("\n===== KET QUA DANH GIA MO HINH =====")

display(bang_danh_gia)

print(f"MAE: {mae:.2f}")

if pd.notna(mape):
    print(f"MAPE: {mape:.2f}%")
else:
    print("MAPE: Khong the tinh do gia tri thuc te bang 0.")


# ---------------------------------------------------------
# 3.2.3. DU BAO 2 TUAN TIEP THEO
# ---------------------------------------------------------

# Du bao tuan thu nhat
forecast_1 = weekly["job_count"].iloc[-3:].mean()

# Du bao tuan thu hai
forecast_2 = (
    weekly["job_count"].iloc[-2:].sum()
    + forecast_1
) / 3

# Tinh ngay bat dau cua 2 tuan tiep theo
last_date = weekly["post_date"].iloc[-1]

forecast_dates = [
    last_date + pd.Timedelta(weeks=1),
    last_date + pd.Timedelta(weeks=2)
]

forecast = pd.DataFrame({
    "post_date": forecast_dates,
    "predicted_job_count": [
        round(forecast_1),
        round(forecast_2)
    ]
})

print("\n===== DU BAO 2 TUAN TIEP THEO =====")

display(forecast)


# ---------------------------------------------------------
# 3.2.4. VE BIEU DO THUC TE VA DU BAO
# ---------------------------------------------------------

plt.figure(figsize=(12, 6))

plt.plot(
    weekly["post_date"],
    weekly["job_count"],
    marker="o",
    label="So tin thuc te"
)

plt.plot(
    forecast["post_date"],
    forecast["predicted_job_count"],
    marker="o",
    linestyle="--",
    color="red",
    label="So tin du bao"
)

plt.title("Du bao so luong tin tuyen dung IT theo tuan")
plt.xlabel("Tuan")
plt.ylabel("So tin tuyen dung")

plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()

forecast_chart = CHART_DIR / "weekly_forecast.png"

plt.savefig(
    forecast_chart,
    dpi=300,
    bbox_inches="tight"
)

print("Da xuat:", forecast_chart)

# Dung show goc de khong lam lech thu tu chart_names
_original_show()


# ---------------------------------------------------------
# 3.2.5. XUAT KET QUA DU BAO
# ---------------------------------------------------------

forecast.to_csv(
    TABLE_DIR / "weekly_forecast.csv",
    index=False,
    encoding="utf-8-sig"
)

bang_danh_gia.to_csv(
    TABLE_DIR / "forecast_model_evaluation.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Da xuat bang du bao va bang danh gia mo hinh.")

In [ ]:
# =========================================================
# PHAN TICH 4: PHAN TICH LUONG
# =========================================================

# Ghep du lieu vai tro va cap bac

luong = jobs.merge(
    dim_role,
    on="role_id",
    how="left"
)

luong = luong.merge(
    dim_level,
    on="level_id",
    how="left"
)


# ---------------------------------------------------------
# 4.1. LUONG TRUNG VI THEO VAI TRO
# ---------------------------------------------------------

luong_theo_vai_tro = (
    luong[luong["salary_avg"].notna()]
    .groupby("role_name")
    .agg(
        luong_trung_vi=("salary_avg", "median"),
        so_tin_cong_khai=("job_id", "nunique")
    )
    .sort_values(
        "luong_trung_vi",
        ascending=False
    )
    .reset_index()
)

print("\n===== LUONG TRUNG VI THEO VAI TRO =====")

display(luong_theo_vai_tro)

plt.figure(figsize=(12, 7))

sns.barplot(
    data=luong_theo_vai_tro,
    x="luong_trung_vi",
    y="role_name",
    color="steelblue"
)

plt.title("Luong trung vi theo vai tro")
plt.xlabel("Luong trung vi (trieu VND/thang)")
plt.ylabel("Vai tro")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 4.2. LUONG TRUNG VI THEO CAP BAC
# ---------------------------------------------------------

luong_theo_cap_bac = (
    luong[luong["salary_avg"].notna()]
    .groupby("level_name")
    .agg(
        luong_trung_vi=("salary_avg", "median"),
        so_tin_cong_khai=("job_id", "nunique")
    )
    .sort_values(
        "luong_trung_vi",
        ascending=False
    )
    .reset_index()
)

print("\n===== LUONG TRUNG VI THEO CAP BAC =====")

display(luong_theo_cap_bac)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=luong_theo_cap_bac,
    x="luong_trung_vi",
    y="level_name",
    color="steelblue"
)

plt.title("Luong trung vi theo cap bac")
plt.xlabel("Luong trung vi (trieu VND/thang)")
plt.ylabel("Cap bac")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 4.3. TY LE TIN TUYEN DUNG CO CONG KHAI LUONG
# ---------------------------------------------------------

tong_so_tin = jobs["job_id"].nunique()

so_tin_cong_khai_luong = (
    jobs.loc[
        jobs["salary_avg"].notna(),
        "job_id"
    ].nunique()
)

so_tin_khong_cong_khai_luong = (
    tong_so_tin - so_tin_cong_khai_luong
)

ty_le_cong_khai_luong = (
    so_tin_cong_khai_luong / tong_so_tin * 100
)

ty_le_khong_cong_khai_luong = (
    so_tin_khong_cong_khai_luong / tong_so_tin * 100
)

bang_cong_khai_luong = pd.DataFrame({
    "trang_thai": [
        "Co cong khai luong",
        "Khong cong khai luong"
    ],
    "so_tin": [
        so_tin_cong_khai_luong,
        so_tin_khong_cong_khai_luong
    ],
    "ty_le": [
        ty_le_cong_khai_luong,
        ty_le_khong_cong_khai_luong
    ]
})

print("\n===== TY LE TIN TUYEN DUNG CO CONG KHAI LUONG =====")

display(bang_cong_khai_luong)

plt.figure(figsize=(8, 6))

plt.pie(
    bang_cong_khai_luong["so_tin"],
    labels=bang_cong_khai_luong["trang_thai"],
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Ty le tin tuyen dung co cong khai luong")

plt.tight_layout()
plt.show()

print(
    f"\nSo tin cong khai luong: "
    f"{so_tin_cong_khai_luong:,}/{tong_so_tin:,}"
)

print(
    f"Ty le cong khai luong: "
    f"{ty_le_cong_khai_luong:.1f}%"
)


# ---------------------------------------------------------
# 4.4. GHI CHU PHAN TICH LUONG
# ---------------------------------------------------------

print("\n===== GHI CHU PHAN TICH LUONG =====")

print(
    f"Co {ty_le_cong_khai_luong:.1f}% tin tuyen dung "
    f"({so_tin_cong_khai_luong:,}/{tong_so_tin:,}) "
    "co cong khai luong."
)

print(
    "Luong trung vi mang tinh tham khao "
    "do so luong tin cong khai luong han che."
)


# =========================================================
# 5. XUAT CAC BANG KET QUA EDA
# =========================================================

tables = {
    "jobs_by_city": tin_theo_thanh_pho,
    "jobs_by_role": tin_theo_vai_tro,
    "top_15_skills": top_15_ky_nang,
    "skills_by_role": bang_ky_nang_theo_vai_tro.reset_index(),
    "skill_requirement_rate": ty_le_ky_nang,
    "recruitment_trend": tin_theo_thoi_gian,
    "salary_by_role": luong_theo_vai_tro,
    "salary_by_level": luong_theo_cap_bac,
    "salary_disclosure": bang_cong_khai_luong
}

for name, table in tables.items():

    table.to_csv(
        TABLE_DIR / f"{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Da xuat bang: {name}.csv")


# =========================================================
# 6. TONG KET
# =========================================================

print("\n" + "=" * 50)
print("HOAN TAT PHAN TICH KHAM PHA DU LIEU")
print("=" * 50)

print(f"Tong so tin tuyen dung: {tong_so_tin:,}")
print(f"Tong so cong ty xac dinh duoc: {tong_so_cong_ty:,}")
print(f"So dong ky nang: {len(fact_skills):,}")
print(f"So tin co du lieu ngay dang: {len(xu_huong):,}")
print(f"So tin co cong khai luong: {so_tin_cong_khai_luong:,}")

print("\nThu muc bieu do:", CHART_DIR)
print("Thu muc bang ket qua:", TABLE_DIR)